# Transcripción y diarización de audio

Pipeline de [`scripts/test_audio_ovh.py`](../scripts/test_audio_ovh.py) paso a paso:

**vídeo → audio → Whisper en OVHcloud (con diarización) → turnos de habla → CSV del contrato → tiempo de habla por hablante**

## 0. Configuración

In [ ]:
import os, json, subprocess
from pathlib import Path
import requests
import pandas as pd

VIDEO = "videos/la-ia-i-el-talent-femeni-ytmp4.savetube.vip.mp4"   # <- vídeo a analizar
LANG = "ca"                                                       # idioma del audio: ca, es, en...

BASE = os.environ.get("OVH_BASE_URL", "https://oai.endpoints.kepler.ai.cloud.ovh.net/v1")
TOKEN = "".join((os.environ.get("OVH_AI_ENDPOINTS_ACCESS_TOKEN") or os.environ.get("OVH_API_TOKEN") or "").split())
ASR_MODEL = os.environ.get("OVH_ASR_MODEL", "whisper-large-v3")
assert TOKEN, "Falta el token: define la variable de entorno OVH_AI_ENDPOINTS_ACCESS_TOKEN"

## 1. Extraer el audio del vídeo

WAV mono a 16 kHz: es el formato con el que trabaja Whisper y pesa menos al subirlo.

In [ ]:
AUDIO = Path(VIDEO).with_suffix(".wav").name
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO, "-vn", "-ac", "1", "-ar", "16000", AUDIO], check=True)
print(f"{AUDIO}: {Path(AUDIO).stat().st_size / 1e6:.1f} MB")

## 2. Transcribir con Whisper en OVHcloud

`diarize=true` pide que, además del texto, se indique qué hablante dice cada fragmento. Guardamos la respuesta cruda en JSON para poder revisarla.

In [ ]:
with open(AUDIO, "rb") as f:
    r = requests.post(f"{BASE}/audio/transcriptions",
                      headers={"Authorization": f"Bearer {TOKEN}"},
                      files={"file": (AUDIO, f)},
                      data={"model": ASR_MODEL, "language": LANG, "response_format": "verbose_json", "diarize": "true"},
                      timeout=180)
assert r.ok, f"Error {r.status_code}: {r.text[:500]}"
res = r.json()

Path(AUDIO).with_suffix(".json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
list(res.keys())

## 3. Turnos de habla

OVHcloud devuelve los turnos con su hablante en la clave `diarization` (si no está, usamos `segments`, sin hablante).

In [ ]:
turnos = res.get("diarization") or res.get("segments", [])

df = pd.DataFrame([{
    "start": float(s.get("start", 0.0)),
    "end": float(s.get("end", 0.0)),
    "speaker": f"Hablante_{s['speaker']}" if isinstance(s.get("speaker"), int) else str(s.get("speaker", "Desconocido")),
    "text": s.get("text", "").strip(),
} for s in turnos])
print(len(df), "turnos")
df.head(10)

## 4. Género de la voz (pendiente)

El script tiene preparado el clasificador `prithivMLmods/Common-Voice-Gender-Detection`, pero todavía no lo aplica a los turnos. Mientras tanto, todos quedan como `no_determinado` para revisión humana.

In [ ]:
df["genero_inferido"] = "no_determinado"

## 5. Exportar el CSV del contrato de datos

Columnas acordadas con el equipo: `tiempo_i`, `tiempo_f`, `transcripcion`, `genero_inferido`.

In [ ]:
contrato = (df.rename(columns={"start": "tiempo_i", "end": "tiempo_f", "text": "transcripcion"})
              [["tiempo_i", "tiempo_f", "transcripcion", "genero_inferido"]].round(2))
contrato.to_csv(Path(AUDIO).with_suffix(".csv").name, index=False, encoding="utf-8")
contrato.head()

## 6. Tiempo de habla por hablante

In [ ]:
dur = (df.end - df.start).clip(lower=0)
tiempos = dur.groupby(df.speaker).sum().sort_values(ascending=False)

resumen = pd.DataFrame({"segundos": tiempos.round(1), "%": (100 * tiempos / tiempos.sum()).round(1)})
print(f"Total voz detectada: {tiempos.sum():.1f} s")
resumen